# Fashion Trend Forecasting - Data and Hypothesis

**Research question:** Can the first 8 weeks of a season's popularity data predict that style's seasonal peak? Does adding early review ratings improve accuracy further?

**Why it matters:** Fashion retailers commit to inventory months before a season begins. A model that forecasts peak demand from the first few weeks of early-season data could reduce overstock, cut markdowns, and improve sell-through rates. Even a modest MAE improvement of 1 - 2 popularity points translates to better stock decisions across thousands of SKUs.

**Data design:** Each style has a hidden annual buzz factor (mid-season boost) that elevates both peak popularity and early review ratings, creating a genuine independent signal in reviews that the trend line alone cannot see.

## Why Ridge, and not something fancier?

With only 48 instances, the smart choice isn't the most powerful model -
it's the one that won't overfit.

- ARIMA / Prophet watch a single signal, so they can't blend in the
  review data - and they want years of history I don't have.
- XGBoost / LightGBM are data-hungry; on 48 rows they'd just memorise
  noise. My Random Forest (Model C) already covers that check with less risk.
- LSTM / Transformer need far more data - a non-starter here.

So a Ridge regression is conditioned on style identity and is regularised
against overfitting - interpretable, stable, and quick to validate. The
Random Forest rides along as a second opinion.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEED    = 42
N_YEARS = 6
PERIOD  = 52          # weeks per year
N_WEEKS = N_YEARS * PERIOD   # 312

DATA_DIR    = Path('data')
RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)

rng = np.random.default_rng(SEED)

## Why synthetic data?

We use fully synthetic and completely deterministic data (seed = 42). This has a concrete reason: we know exactly what the true peak popularity value is for every style and every year - meaning we can honestly verify whether the models capture something real, not just noise.

In a real-world project the next step would be replacing the synthetic CSV files with real data (Google Trends + product reviews) without changing any code.

In [8]:
STYLES = {
    'oversized blazer': {'base': 40, 'slope':  0.060, 'amp':  8, 'peak_week':  8, 'fad': (120, 14, 16), 'noise': 3},
    'wide-leg jeans':   {'base': 33, 'slope':  0.110, 'amp':  6, 'peak_week':  2, 'fad': None,          'noise': 3},
    'cargo pants':      {'base': 30, 'slope':  0.045, 'amp':  5, 'peak_week':  4, 'fad': (150,  9, 26), 'noise': 4},
    'ballet flats':     {'base': 45, 'slope':  0.030, 'amp':  7, 'peak_week': 14, 'fad': None,          'noise': 3},
    'puffer jacket':    {'base': 50, 'slope':  0.000, 'amp': 30, 'peak_week': 50, 'fad': None,          'noise': 3},
    'slip dress':       {'base': 42, 'slope': -0.020, 'amp': 22, 'peak_week': 26, 'fad': None,          'noise': 3},
    'trench coat':      {'base': 48, 'slope': -0.050, 'amp': 12, 'peak_week': 40, 'fad': None,          'noise': 3},
    'chunky sneakers':  {'base': 56, 'slope': -0.080, 'amp':  5, 'peak_week':  1, 'fad': None,          'noise': 4},
}
STYLE_NAMES = list(STYLES.keys())

Hidden annual buzz factor - peaks at week 26 of each year.
Detectable in early review ratings, but NOT in the first 8 weeks of the trend line.

In [9]:
BUZZ_STD    = 10.0
buzz_matrix = rng.normal(0, BUZZ_STD, size=(len(STYLES), N_YEARS))  # shape (8, 6)

## Style design

The eight styles form four archetypes:
-  Rising - wide-leg jeans, blazer, cargo -  Slope is > 0 
- Stable - puffer jacket - Slope is around ≈ 0 
- Fading - trench coat, chunky sneakers - Slope is  < 0 
- Viral spike - cargo pants, blazer - fad Gaussian 

The Buzz factor is a hidden annual variable (± ~10 pts) that amplifies or reduces the mid-season peak. Early buyers sense it and leave better/worse reviews - but the trend line in weeks 1-8 does not yet show it, because the buzz peak is centred around week 26.

In [10]:
def generate_style_series(params, buzz_yrs):
    """Generate 312-week popularity series. Buzz peaks at week 26 of each year."""
    t      = np.arange(N_WEEKS)
    trend  = params['base'] + params['slope'] * t
    season = params['amp'] * np.cos(2 * np.pi * (t - params['peak_week']) / PERIOD)
    fad    = np.zeros(N_WEEKS)
    if params['fad']:
        c, w, h = params['fad']
        fad = h * np.exp(-((t - c)**2) / (2 * w**2))
    buzz = np.zeros(N_WEEKS)
    for y in range(N_YEARS):
        center = y * PERIOD + 25           # week 26 (0-indexed: 25)
        buzz  += buzz_yrs[y] * np.exp(-((t - center)**2) / (2 * 6**2))
    noise = rng.normal(0, params['noise'], size=N_WEEKS)
    return np.clip(trend + season + fad + buzz + noise, 0, 100)

dates      = pd.date_range('2021-01-03', periods=N_WEEKS, freq='W')
all_series = {}
rows       = {'date': dates}
for i, (style, params) in enumerate(STYLES.items()):
    s = generate_style_series(params, buzz_matrix[i])
    all_series[style] = s
    rows[style] = s

style_trends = pd.DataFrame(rows)
style_trends['year']         = style_trends.index // PERIOD
style_trends['week_in_year'] = style_trends.index % PERIOD + 1
style_trends.to_csv(DATA_DIR / 'trends_synthetic.csv', index=False)
print(f'trends_synthetic.csv written — {style_trends.shape}')
style_trends.head()

trends_synthetic.csv written — (312, 11)


,date,oversized blazer,wide-leg jeans,cargo pants,ballet flats,puffer jacket,slip dress,trench coat,chunky sneakers,year,week_in_year
0,2021-01-03,45.782517,42.714374,44.070765,46.544135,80.663960,19.130793,48.443907,50.722287,0,1
1,2021-01-10,46.744214,39.550788,36.982771,45.903136,84.273877,18.740786,47.421329,56.404845,0,2
2,2021-01-17,44.354876,38.624272,32.556240,46.147980,81.020313,19.619241,42.086339,66.653215,0,3
3,2021-01-24,48.153189,36.681343,42.797456,38.997775,70.174259,24.210161,43.151820,63.925025,0,4
4,2021-01-31,46.745994,44.550090,33.771074,53.788134,70.845340,20.068562,42.330444,65.610751,0,5
